# Create Office of Naval Research (ONR) Awards

The **Office of Naval Research** funds a large basic- and applied-research portfolio; its `N00014-...` grant numbers are heavily cited in papers, so this ingest is valuable as direct **work<->award linkage**.

**Source — USAspending.gov (`scripts/local/onr_to_s3.py`).** ONR is not cleanly exposed as a USAspending awarding sub-tier (DoD/Navy office fields are null on assistance awards), so the scraper keys off ONR's award-id prefix: a keyword search on `N00014` within assistance awards, keeping only ids that start with `N00014`. The paginated search caps at ~10k records, so the pull is split by year and de-duped by award id.

**Awarding body funder:** Office of Naval Research — funder_id **4320337345** (US; `F4320*` ⇒ Path A, in `openalex.common.funder`).

**Schema choices (ORG-LEVEL GRANT pattern — same shape as the EPA/AHRQ/CDC USAspending ingests):**
- `funder_award_id` = the USAspending Award ID (the `N00014-...` FAIN). `display_name` = the award Description.
- `amount` = Award Amount (**USD**; 0/negative → NULL per §6.7). `start_date`/`end_date` = period of performance (real dates); future-year cap applied.
- `lead_investigator` = **no named PI** (USAspending exposes the recipient organisation, not an individual): given/family/orcid NULL, `affiliation.name` = recipient organisation, `country` = US. `co_lead`/`investigators` NULL.
- `funding_type` = grant; `funder_scheme`/`description` NULL.

**S3:** `s3a://openalex-ingest/awards/onr/onr_projects.parquet`  ·  **provenance:** `usaspending_onr`


## Step 1: Staging table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.onr_raw
USING delta AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/onr/onr_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.onr_raw;


## Step 1.5: Inspect raw + coverage

Expected: ~16,000 ONR (N00014) awards. funder_award_id / institution / start_date ~100%; amount ~99% (USD). title (Description) ~100%.


In [ ]:
%sql
DESCRIBE openalex.awards.onr_raw;


In [ ]:
%sql
SELECT funder_award_id, institution_name, amount, start_date, end_date, SUBSTRING(title,1,50) AS title
FROM openalex.awards.onr_raw LIMIT 8;


In [ ]:
%sql
SELECT COUNT(*) AS total, COUNT(DISTINCT funder_award_id) AS distinct_ids,
       COUNT(title) AS has_title, COUNT(amount) AS has_amount,
       COUNT(institution_name) AS has_inst, COUNT(start_date) AS has_start,
       SUM(CASE WHEN funder_award_id LIKE 'N00014%' THEN 0 ELSE 1 END) AS not_n00014
FROM openalex.awards.onr_raw;


## Step 1.6: Fail-fast — verify ONR funder row exists (Path A)

`4320337345` is `F4320*`, so it MUST be present in `openalex.common.funder`. Must return exactly 1 row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320337345;


## Step 2: Transform to award schema

ORG-LEVEL GRANT: no named PI (affiliation = recipient organisation, country US). amount USD; 0/negative → NULL (§6.7). Future-year cap (§2.3).


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.onr_awards
USING delta AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320337345   -- Office of Naval Research
),
base AS (
    SELECT
        n.funder_award_id, n.title, n.institution_name, n.country,
        TRY_CAST(n.amount AS DOUBLE) AS amt, n.currency,
        TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS sd,
        TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS ed,
        f.funder_id, f.display_name AS f_name, f.ror_id, f.doi AS f_doi
    FROM openalex.awards.onr_raw n
    CROSS JOIN funder_resolved f
    WHERE n.funder_award_id IS NOT NULL
)
SELECT
    abs(xxhash64(CONCAT(TRY_CAST(funder_id AS STRING), ':', LOWER(funder_award_id)))) % 9000000000 AS id,
    title AS display_name,
    CAST(NULL AS STRING) AS description,
    funder_id,
    funder_award_id,
    CASE WHEN amt > 0 THEN amt ELSE NULL END AS amount,
    CASE WHEN amt > 0 THEN currency ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(funder_id AS STRING)) AS id,
        f_name AS display_name, ror_id AS ror_id, f_doi AS doi
    ) AS funder,
    'grant' AS funding_type,
    CAST(NULL AS STRING) AS funder_scheme,
    'usaspending_onr' AS provenance,
    sd AS start_date,
    ed AS end_date,
    CASE WHEN YEAR(sd) > YEAR(current_date()) + 1 THEN NULL ELSE YEAR(sd) END AS start_year,
    CASE WHEN YEAR(sd) > YEAR(current_date()) + 1 THEN NULL ELSE YEAR(ed) END AS end_year,
    CASE WHEN institution_name IS NOT NULL THEN struct(
        CAST(NULL AS STRING) AS given_name,
        CAST(NULL AS STRING) AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            institution_name AS name,
            country AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) ELSE NULL END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CAST(NULL AS STRING) AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(TRY_CAST(funder_id AS STRING), ':', LOWER(funder_award_id)))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM base;


## Step 3: Insert into openalex_awards_raw at priority 171


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'usaspending_onr' AND priority = 171;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    171 AS priority  -- Office of Naval Research
FROM openalex.awards.onr_awards;


## Step 6: Verification

§6.7 amount **present** (Award Amount, USD; ~99%). Org-level grant: lead_investigator has NULL person + recipient affiliation. display_name / start_year ~100%.


In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.onr_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.onr_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total, COUNT(display_name) AS has_title, COUNT(amount) AS has_amount,
    COUNT(lead_investigator.affiliation.name) AS has_institution, COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    collect_set(currency) AS currencies, ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.onr_awards;


In [ ]:
%sql
SELECT MIN(start_year) AS first_year, MAX(start_year) AS last_year, COUNT(*) AS total,
       COUNT(DISTINCT id) AS distinct_ids,
       SUM(CASE WHEN start_year > YEAR(current_date())+1 THEN 1 ELSE 0 END) AS future_year_rows,
       SUM(CASE WHEN funder_award_id LIKE 'N00014%' THEN 0 ELSE 1 END) AS not_n00014
FROM openalex.awards.onr_awards;


In [ ]:
%sql
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.onr_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
SELECT lead_investigator.affiliation.name AS institution, COUNT(*) AS n, ROUND(SUM(amount)/1e6,1) AS total_musd
FROM openalex.awards.onr_awards
GROUP BY lead_investigator.affiliation.name ORDER BY n DESC LIMIT 10;
